In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install fastapi uvicorn[standard] pyngrok lime joblib transformers torch

In [ ]:
!pip -q install pdfplumber PyPDF2 python-docx


In [ ]:
%%writefile extractor.py
import io
import re


def normalize_text(s: str) -> str:
    s = (s or "").replace("\r\n", "\n").replace("\r", "\n")
    while "\n\n\n" in s:
        s = s.replace("\n\n\n", "\n\n")
    return s.strip()


def extract_text_from_pdf(file_bytes: bytes) -> str:
    try:
        import pdfplumber
        parts = []
        with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
            for page in pdf.pages:
                t = page.extract_text() or ""
                if t.strip():
                    parts.append(t)
        return normalize_text("\n\n".join(parts))
    except Exception:
        try:
            import PyPDF2
            reader = PyPDF2.PdfReader(io.BytesIO(file_bytes))
            parts = []
            for p in reader.pages:
                t = p.extract_text() or ""
                if t.strip():
                    parts.append(t)
            return normalize_text("\n\n".join(parts))
        except Exception:
            return ""

def extract_text_from_docx(file_bytes: bytes) -> str:
    try:
        import docx
        doc = docx.Document(io.BytesIO(file_bytes))
        parts = [p.text for p in doc.paragraphs if p.text and p.text.strip()]
        return normalize_text("\n".join(parts))
    except Exception:
        return ""

def extract_text_from_txt(file_bytes: bytes) -> str:
    try:
        return normalize_text(file_bytes.decode("utf-8", errors="ignore"))
    except Exception:
        return normalize_text(file_bytes.decode("latin-1", errors="ignore"))

def file_bytes_to_text(filename: str, content: bytes) -> str:
    fn = (filename or "").lower()
    if fn.endswith(".pdf"):
        return extract_text_from_pdf(content)
    if fn.endswith(".docx"):
        return extract_text_from_docx(content)
    if fn.endswith(".txt"):
        return extract_text_from_txt(content)
    return ""


SECTION_START = [
    "history", "findings", "impression", "conclusion", "report", "discussion", "assessment", "opinion"
]

FOOTER_TRIGGERS = [
    "thank you", "computer generated", "no signature", "signature", "signed",
    "consultant", "radiologist", "interventional", "fcps", "frcr", "mrcp", "fellowship"
]

ADMIN_HINTS = [
    "v-no", "v no", "radiology", "radiologist", "hospital", "patient name", "patient age",
    "patient r. date", "patient r date", "referred", "ref:", "mrn", "id", "date:", "clinic"
]

def extract_clinical_text(text: str) -> str:
    t = normalize_text(text)
    if not t:
        return ""

    lines = [ln.strip() for ln in t.split("\n")]
    non_empty = [ln for ln in lines if ln]

    # If already short/description-like, keep (light cleanup)
    if len(" ".join(non_empty)) < 650:
        return re.sub(r"\s+", " ", " ".join(non_empty)).strip()

    # 1) Remove admin/header lines
    cleaned_lines = []
    for i, ln in enumerate(lines):
        low = ln.lower().strip()
        if not low:
            cleaned_lines.append("")
            continue


        if any(k in low for k in ["patient name", "patient age", "patient r", "referred", "mrn", "v-no", "v no"]):
            continue

        if i < 30 and any(h in low for h in ADMIN_HINTS):
            continue

        cleaned_lines.append(ln)

    t2 = normalize_text("\n".join(cleaned_lines))
    low2 = t2.lower()


    start_idx = None
    for key in SECTION_START:
        m = re.search(rf"\b{re.escape(key)}\b\s*[:\-]?", low2)
        if m:
            start_idx = m.start()
            break
    if start_idx is not None:
        t2 = t2[start_idx:].strip()
        low2 = t2.lower()


    end_idx = None
    for trig in FOOTER_TRIGGERS:
        m = re.search(rf"\b{re.escape(trig)}\b", low2)
        if m and m.start() > 250:
            end_idx = m.start()
            break
    if end_idx is not None:
        t2 = t2[:end_idx].strip()

    if len(t2) < 180:
        return re.sub(r"\s+", " ", " ".join(non_empty)).strip()

    return re.sub(r"[ \t]+", " ", t2).strip()


In [ ]:
%%writefile lime_engine.py
import numpy as np
import torch
import torch.nn.functional as F
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer

import extractor

LOAD_PATH = "/content/drive/MyDrive/apimodel"

print("Loading model/tokenizer/label encoder...")
tokenizer = AutoTokenizer.from_pretrained(LOAD_PATH, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(LOAD_PATH, local_files_only=True)
label_encoder = joblib.load(f"{LOAD_PATH}/label_encoder.joblib")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

class_names = list(label_encoder.classes_)
explainer = LimeTextExplainer(class_names=class_names, random_state=42, bow=False)

def lime_predictor(texts):
    all_probs = []
    for i in range(0, len(texts), 16):
        batch = texts[i:i + 16]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs.detach().cpu().numpy())
    return np.vstack(all_probs)

def apply_guardrails(raw_text: str, raw_probs: np.ndarray, pred_idx: int):
    text_lower = (raw_text or "").lower()

    ischemic = ["infarct", "stroke", "mca", "aphasia", "hemiparesis", "ischemia"]
    hemorrhage = ["hemorrhage", "hematoma", "bleed", "blood", "thalamic", "intraparenchymal"]
    ms_markers = ["ms", "multiple sclerosis", "demyelinating", "demyelination", "dawson", "ovoid", "plaques"]

    surgery = ["resection", "post-op", "cavity", "craniotomy", "surgical", "scarring", "gliosis"]
    tumor = ["mass", "neoplasm", "lesion", "metastasis", "malignancy", "enhancing nodule", "gbm", "glioma"]

    is_non_tumor_pathology = any(w in text_lower for w in ischemic + hemorrhage + ms_markers)
    has_conflicting_surgery = any(w in text_lower for w in surgery)
    has_conflicting_tumor = any(w in text_lower for w in tumor)

    overridden = False
    conf = float(raw_probs[pred_idx])

    if is_non_tumor_pathology and not (has_conflicting_surgery or has_conflicting_tumor):
        no_tumor_idx = -1
        for i, name in enumerate(label_encoder.classes_):
            nm = name.lower()
            if "no" in nm or "normal" in nm:
                no_tumor_idx = i
                break

        if no_tumor_idx != -1:
            overridden = True
            pred_idx = no_tumor_idx
            second_best = float(np.sort(raw_probs)[-2]) if len(raw_probs) > 1 else 0.0
            allowed_sum = float(raw_probs[no_tumor_idx]) + (second_best * 0.05)
            conf = float(raw_probs[no_tumor_idx] / allowed_sum) if allowed_sum > 0 else 0.98

    return pred_idx, conf, overridden

def predict(text: str):

    clean_text = extractor.extract_clinical_text(text)

    inputs = tokenizer(clean_text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)[0].detach().cpu().numpy()

    pred_idx = int(np.argmax(probs))
    pred_idx, conf, overridden = apply_guardrails(clean_text, probs, pred_idx)

    label = label_encoder.inverse_transform([pred_idx])[0]
    return clean_text, {
        "class_id": pred_idx,
        "label": str(label),
        "confidence": float(conf),
        "overridden": bool(overridden),
        "all_probs": probs.tolist()
    }

def explain(text: str, num_features: int = 10, num_samples: int = 250, max_chars: int = 1600):
    clean_text = extractor.extract_clinical_text(text)
    t = clean_text[:int(max_chars)] if max_chars and max_chars > 0 else clean_text


    inputs = tokenizer(t, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)[0].detach().cpu().numpy()

    pred_idx = int(np.argmax(probs))
    pred_idx, conf, overridden = apply_guardrails(t, probs, pred_idx)
    label = label_encoder.inverse_transform([pred_idx])[0]

    pred = {
        "class_id": pred_idx,
        "label": str(label),
        "confidence": float(conf),
        "overridden": bool(overridden),
        "all_probs": probs.tolist()
    }

    exp = explainer.explain_instance(
        t, lime_predictor,
        num_features=int(num_features),
        labels=(pred_idx,),
        num_samples=int(num_samples)
    )

    token_weights = exp.as_list(label=pred_idx)
    explanation = [{"token": a, "weight": float(b)} for a, b in token_weights]

    return clean_text, {
        "prediction": pred,
        "explanation": {
            "method": "LIME",
            "label_index": pred_idx,
            "label_name": pred["label"],
            "num_features": int(num_features),
            "num_samples": int(num_samples),
            "max_chars": int(max_chars),
            "used_chars": len(t),
            "token_weights": explanation
        }
    }


In [ ]:
%%writefile app.py
import uuid
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

import extractor
import lime_engine

app = FastAPI(title="BioBERT MRI API (Predict + LIME Explain)", version="1.0-modular")

# CORS (Flutter / ngrok safe)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class PredictRequest(BaseModel):
    text: str

class ExplainRequest(BaseModel):
    text: str
    num_features: int = 10
    num_samples: int = 250
    max_chars: int = 1600

@app.get("/health")
def health():
    return {"status": "ok", "device": str(lime_engine.device)}

@app.post("/predict")
def predict_text(req: PredictRequest):
    if not req.text or len(req.text.strip()) < 10:
        raise HTTPException(status_code=400, detail="Text is empty/too short.")

    clean_text, result = lime_engine.predict(req.text)

    if not clean_text or len(clean_text) < 50:
        raise HTTPException(
            status_code=422,
            detail="Clinical text not detected. Please paste findings/description only."
        )

    return {
        "request_id": str(uuid.uuid4()),
        "source": {"type": "text"},
        "extracted_text": clean_text,
        "preprocess": {"mode": "cleaned", "cleaned_chars": len(clean_text)},
        **result
    }

@app.post("/explain")
def explain_text(req: ExplainRequest):
    if not req.text or len(req.text.strip()) < 10:
        raise HTTPException(status_code=400, detail="Text is empty/too short.")

    clean_text, out = lime_engine.explain(req.text, req.num_features, req.num_samples, req.max_chars)

    if not clean_text or len(clean_text) < 50:
        raise HTTPException(
            status_code=422,
            detail="Clinical text not detected. Please paste findings/description only."
        )

    return {
        "request_id": str(uuid.uuid4()),
        "source": {"type": "text"},
        "extracted_text": clean_text,
        "preprocess": {"mode": "cleaned", "cleaned_chars": len(clean_text)},
        **out
    }

@app.post("/predict_file")
async def predict_file(file: UploadFile = File(...)):
    content = await file.read()
    raw = extractor.file_bytes_to_text(file.filename, content)

    if not raw or len(raw) < 50:
        raise HTTPException(
            status_code=422,
            detail="Could not extract readable text from file. Please copy-paste report text."
        )

    clean_text, result = lime_engine.predict(raw)

    if not clean_text or len(clean_text) < 50:
        raise HTTPException(
            status_code=422,
            detail="Clinical text not detected. Please paste findings/description only."
        )

    return {
        "request_id": str(uuid.uuid4()),
        "source": {"filename": file.filename, "type": "file"},
        "extracted_text": clean_text,
        "preprocess": {
            "mode": "cleaned",
            "raw_chars": len(raw),
            "cleaned_chars": len(clean_text)
        },
        **result
    }

@app.post("/explain_file")
async def explain_file(
    file: UploadFile = File(...),
    num_features: int = 10,
    num_samples: int = 250,
    max_chars: int = 1600
):
    content = await file.read()
    raw = extractor.file_bytes_to_text(file.filename, content)

    if not raw or len(raw) < 50:
        raise HTTPException(
            status_code=422,
            detail="Could not extract readable text from file. Please copy-paste report text."
        )

    clean_text, out = lime_engine.explain(raw, num_features, num_samples, max_chars)

    if not clean_text or len(clean_text) < 50:
        raise HTTPException(
            status_code=422,
            detail="Clinical text not detected. Please paste findings/description only."
        )

    return {
        "request_id": str(uuid.uuid4()),
        "source": {"filename": file.filename, "type": "file"},
        "extracted_text": clean_text,
        "preprocess": {
            "mode": "cleaned",
            "raw_chars": len(raw),
            "cleaned_chars": len(clean_text)
        },
        **out
    }


In [ ]:
#!uvicorn app:app --host 0.0.0.0 --port 8000 --reload

In [ ]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &

In [ ]:
!pkill -f uvicorn

In [ ]:
!tail -n 20 uvicorn.log

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("please add your ngrok token")

tunnel = ngrok.connect(8000)
print("Public URL:", tunnel.public_url)


In [ ]:
import requests
print(requests.get(tunnel.public_url + "/health").json())

In [ ]:
import requests

base = "https://uncushioned-subsuperficially-theressa.ngrok-free.dev"
headers = {
    "ngrok-skip-browser-warning": "true",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

def post_predict(text):
    r = requests.post(base + "/predict", json={"text": text}, headers=headers, timeout=120)
    print("\n--- STATUS:", r.status_code, "---")
    print("HEADERS content-type:", r.headers.get("content-type"))
    print("BODY (first 300 chars):", r.text[:300])
    if r.headers.get("content-type", "").startswith("application/json"):
        return r.json()
    # fallback: try json anyway
    try:
        return r.json()
    except Exception:
        return None

# Health check first
h = requests.get(base + "/health", headers=headers, timeout=60)
print("HEALTH STATUS:", h.status_code)
print("HEALTH BODY:", h.text[:200])

a = post_predict("Impression: No tumor. Normal MRI.")
b = post_predict("Impression: Large enhancing mass with edema, glioblastoma likely.")

print("\nA JSON:", a)
print("B JSON:", b)


In [ ]:
!ps aux | grep uvicorn
!tail -n 60 uvicorn.log


In [ ]:
!tail -n 80 uvicorn.log